In [1]:
# ---------------------------------------------------------
# SILVER TRANSFORMATION STEP (LOAD BRONZE → START CLEANING)
# Purpose: Load raw Bronze files into DataFrames for
# cleaning and standardisation. This is the entry point
# for the Silver layer.
# ---------------------------------------------------------

import pandas as pd

# Load Bronze CSVs into DataFrames
df_listings = pd.read_csv("data/bronze/listings_full.csv", low_memory=False)
df_reviews = pd.read_csv("data/bronze/reviews_full.csv", low_memory=False)
df_calendar = pd.read_csv("data/bronze/calendar.csv", low_memory=False)
df_neighbourhoods = pd.read_csv("data/bronze/neighbourhoods.csv")
# GeoJSON stays as-is for now, can load with geopandas later if needed

print("✅ Loaded Bronze into DataFrames")
print("Listings shape:", df_listings.shape)
print("Reviews shape:", df_reviews.shape)
print("Calendar shape:", df_calendar.shape)
print("Neighbourhoods shape:", df_neighbourhoods.shape)

✅ Loaded Bronze into DataFrames
Listings shape: (94559, 79)
Reviews shape: (1932265, 6)
Calendar shape: (34512421, 7)
Neighbourhoods shape: (33, 2)


In [2]:
# ---------------------------------------------------------
# SILVER STEP: CLEAN FULL LISTINGS (no NLP fields)
# Purpose: Transform raw full listings into a clean Silver
# table:
#   - Drop obvious junk (URLs, empty cols, metadata, free-text)
#   - Parse dates
#   - Convert numeric-like strings (price, rates) to numeric
#   - Convert object columns to category where appropriate
#   - Drop duplicates and report
#   - Save to Silver as Parquet
# ---------------------------------------------------------

import pandas as pd
import os

os.makedirs("data/silver", exist_ok=True)

# Load full listings from Bronze
df_listings_full = pd.read_csv("data/bronze/listings_full.csv", low_memory=False)

df_listings_clean = df_listings_full.copy()

# Drop obvious junk: URLs, empty cols, metadata, free-text
drop_cols = [
    # URLs
    "listing_url", "picture_url", "host_url",
    "host_thumbnail_url", "host_picture_url",
    # Empty columns
    "neighbourhood_group_cleansed", "calendar_updated", "license",
    # Metadata
    "scrape_id", "last_scraped", "source", "calendar_last_scraped",
    # Free-text (NLP-heavy, not needed now)
    "description", "neighborhood_overview", "name", "host_about", "neighbourhood"
]
df_listings_clean = df_listings_clean.drop(columns=drop_cols, errors="ignore")

# Parse date columns
date_cols = ["host_since", "first_review", "last_review"]
for col in date_cols:
    if col in df_listings_clean.columns:
        df_listings_clean[col] = pd.to_datetime(df_listings_clean[col], errors="coerce")

# Convert price to numeric
if "price" in df_listings_clean.columns:
    df_listings_clean["price"] = (
        df_listings_clean["price"]
        .replace(r"[\$,]", "", regex=True)
        .astype(float)
    )

# Convert percentage-like strings (response/acceptance rates) to numeric
for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in df_listings_clean.columns:
        df_listings_clean[col] = (
            df_listings_clean[col]
            .str.replace("%", "", regex=False)
            .astype(float)
        )

# Convert object columns (non-text heavy) to category
text_heavy = ["amenities", "bathrooms_text"]  # keep these as text for now
for col in df_listings_clean.select_dtypes(include="object").columns:
    if col not in text_heavy:
        df_listings_clean[col] = df_listings_clean[col].astype("category")

# Drop duplicates and report
before = len(df_listings_clean)
df_listings_clean = df_listings_clean.drop_duplicates()
after = len(df_listings_clean)

print(f"✅ Listings cleaned: {after:,} rows (dropped {before - after:,} duplicates)")

# Save to Silver
df_listings_clean.to_parquet("data/silver/listings_clean.parquet", index=False)

# Quick NA summary
print("\nNA counts per column:")
print(df_listings_clean.isna().sum().sort_values(ascending=False).head(20))

✅ Listings cleaned: 94,559 rows (dropped 0 duplicates)

NA counts per column:
host_neighbourhood             48827
beds                           34277
bathrooms                      34244
price                          34218
estimated_revenue_l365d        34218
host_response_rate             34143
host_response_time             34143
host_acceptance_rate           27194
review_scores_location         24287
review_scores_value            24287
review_scores_checkin          24286
review_scores_communication    24263
review_scores_accuracy         24257
review_scores_cleanliness      24251
review_scores_rating           24242
last_review                    24242
reviews_per_month              24242
first_review                   24242
host_location                  22577
bedrooms                       12835
dtype: int64


In [3]:
# ---------------------------------------------------------
# SILVER STEP: CLEAN FULL REVIEWS
# Purpose: Transform raw full reviews into a clean Silver
# table:
#   - Drop free-text comments (no NLP planned)
#   - Parse date column
#   - Ensure IDs are integers
#   - Convert reviewer_name to category
#   - Drop duplicates and report
#   - Save to Silver as Parquet
# ---------------------------------------------------------

import pandas as pd
import os

os.makedirs("data/silver", exist_ok=True)

# Load full reviews from Bronze
df_reviews_full = pd.read_csv("data/bronze/reviews_full.csv", low_memory=False)

df_reviews_clean = df_reviews_full.copy()

# Drop comments (free text)
df_reviews_clean = df_reviews_clean.drop(columns=["comments"], errors="ignore")

# Parse date column
df_reviews_clean["date"] = pd.to_datetime(df_reviews_clean["date"], errors="coerce")

# Ensure IDs are integers (nullable Int64 to allow NAs if any)
for col in ["listing_id", "id", "reviewer_id"]:
    df_reviews_clean[col] = pd.to_numeric(df_reviews_clean[col], errors="coerce").astype("Int64")

# Convert reviewer_name to category
if "reviewer_name" in df_reviews_clean.columns:
    df_reviews_clean["reviewer_name"] = df_reviews_clean["reviewer_name"].astype("category")

# Drop duplicates and report
before = len(df_reviews_clean)
df_reviews_clean = df_reviews_clean.drop_duplicates()
after = len(df_reviews_clean)

print(f"✅ Reviews cleaned: {after:,} rows (dropped {before - after:,} duplicates)")

# Save to Silver
df_reviews_clean.to_parquet("data/silver/reviews_clean.parquet", index=False)

# Quick NA summary
print("\nNA counts per column:")
print(df_reviews_clean.isna().sum())

✅ Reviews cleaned: 1,932,265 rows (dropped 0 duplicates)

NA counts per column:
listing_id       0
id               0
date             0
reviewer_id      0
reviewer_name    2
dtype: int64


In [4]:
# ---------------------------------------------------------
# SILVER STEP: CLEAN CALENDAR
# Purpose: Transform raw calendar data into a clean Silver
# table:
#   - Parse date column
#   - Convert available to boolean
#   - Convert price and adjusted_price to numeric
#   - Ensure listing_id is integer
#   - Drop duplicates and report
#   - Save to Silver as Parquet
# ---------------------------------------------------------

import pandas as pd
import os

os.makedirs("data/silver", exist_ok=True)

# Load calendar from Bronze
df_calendar = pd.read_csv("data/bronze/calendar.csv", low_memory=False)

df_calendar_clean = df_calendar.copy()

# Parse date column
df_calendar_clean["date"] = pd.to_datetime(df_calendar_clean["date"], errors="coerce")

# Convert available to boolean
df_calendar_clean["available"] = df_calendar_clean["available"].map({"t": True, "f": False})

# Convert price and adjusted_price to numeric
for col in ["price", "adjusted_price"]:
    if col in df_calendar_clean.columns:
        df_calendar_clean[col] = (
            df_calendar_clean[col]
            .replace(r"[\$,]", "", regex=True)
            .astype(float)
        )

# Ensure listing_id is integer (nullable Int64 to allow NAs if any)
df_calendar_clean["listing_id"] = pd.to_numeric(
    df_calendar_clean["listing_id"], errors="coerce"
).astype("Int64")

# Drop duplicates and report
before = len(df_calendar_clean)
df_calendar_clean = df_calendar_clean.drop_duplicates()
after = len(df_calendar_clean)

print(f"✅ Calendar cleaned: {after:,} rows (dropped {before - after:,} duplicates)")

# Save to Silver
df_calendar_clean.to_parquet("data/silver/calendar_clean.parquet", index=False)

# Quick NA summary
print("\nNA counts per column:")
print(df_calendar_clean.isna().sum())

✅ Calendar cleaned: 34,512,421 rows (dropped 0 duplicates)

NA counts per column:
listing_id               0
date                     0
available                0
price                    0
adjusted_price    34499281
minimum_nights        2103
maximum_nights        2103
dtype: int64


In [ ]:
# Load Silver tables
df_listings = pd.read_parquet("data/silver/listings_clean.parquet")
df_reviews = pd.read_parquet("data/silver/reviews_clean.parquet")
df_calendar = pd.read_parquet("data/silver/calendar_clean.parquet")

# Unique listing IDs
listings_ids = set(df_listings["id"])
reviews_ids = set(df_reviews["listing_id"].dropna())
calendar_ids = set(df_calendar["listing_id"].dropna())

# Coverage checks
print("🧾 Listings total:", len(listings_ids))
print("🗣️ Reviews linked:", len(reviews_ids & listings_ids))
print("📅 Calendar linked:", len(calendar_ids & listings_ids))

# Or as percentages
print("Reviews coverage:", len(reviews_ids & listings_ids) / len(reviews_ids))
print("Calendar coverage:", len(calendar_ids & listings_ids) / len(calendar_ids))